# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields by `@id`.
record_sets = []
if hasattr(metadata, 'recordSets'):
    record_sets = metadata.recordSets
elif hasattr(metadata, 'recordSet'):  # schema can use singular or plural
    record_sets = metadata.recordSet

if not record_sets:
    # Try to find record sets by walking the metadata object for Croissant
    warnings.warn("No record sets found via metadata. Attempting to enumerate from dataset.records().")
    print("Trying first dataset.records() to list available record set @ids:")
    available_recordset_ids = dataset.available_record_sets()
    print("Available record set @ids:", available_recordset_ids)
    record_sets = available_recordset_ids
else:
    def get_record_set_id(rs):
        return rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', str(rs))
    record_sets = [get_record_set_id(rs) for rs in record_sets]
    print('Record sets (@id):')
    for r in record_sets:
        print(f"  - {r}")

# Output all fields inside each record set
print("\nFields for each record set:")
for rsid in record_sets:
    try:
        fields = dataset.field_ids(record_set=rsid)
        print(f"- Record set {rsid}: {fields}")
    except Exception as e:
        print(f"  Could not extract fields from {rsid}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Use the first available record set for demonstration
# If you want a different record set, change its @id below
assert record_sets, "No record sets found!"
main_record_set_id = record_sets[0]

# Extract to DataFrame for all fields
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Record set '@id': {main_record_set_id}")
print(f"Number of records: {len(df)}")
print(f"Available fields (column @ids):\n{list(df.columns)}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

In [ ]:
# Attempt to auto-detect a numeric field for exploration
import numpy as np

numeric_field = None
for col in df.columns:
    # Try integer/float detection
    col_sample = pd.to_numeric(df[col], errors='coerce')
    num_valid = col_sample.notnull().sum()
    if num_valid > 0 and num_valid > 0.75*len(df):
        numeric_field = col
        break

if numeric_field is None:
    raise ValueError("No numeric field detected for EDA.")
print(f"Using numeric field for EDA: {numeric_field}")

threshold = np.nanpercentile(pd.to_numeric(df[numeric_field], errors='coerce'), 75)

# Filtering
df_num = df.copy()
df_num[numeric_field] = pd.to_numeric(df_num[numeric_field], errors='coerce')
filtered_df = df_num[df_num[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a group/categorical field
group_field = None
for col in df.columns:
    if col == numeric_field:
        continue
    # Heuristic: string/object, <50 unique values
    if df[col].dtype == object and df[col].nunique() < 30:
        group_field = col
        break

if group_field:
    print(f"\nGrouping by: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    print(grouped_df.head())
else:
    print("No suitable group/categorical field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution
plt.figure(figsize=(7, 4))
sns.histplot(df_num[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot grouped by group_field if available
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df_num, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook provided an overview and first-pass EDA of the FAIR² dataset: _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_.

- We loaded and inspected the dataset's metadata and schema via its Croissant description.
- We enumerated available record sets and fields using the mlcroissant API with all references by their `@id`s.
- We extracted tabular data from the primary record set, performed basic filtering and normalization on a numeric field, and produced simple visualizations.  
- For more in-depth biological/biochemical interpretation, see the [FAIR² package source](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and accompanying dataset publication.
